In [1]:
import torch
import json
from pathlib import Path
from collections import Counter
import time
import math

# Find project root
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data"

# Load vocabulary
with open(DATA_DIR / "vocab.json", "r", encoding="utf-8") as f:
    word_to_id = json.load(f)

id_to_word = {idx: word for word, idx in word_to_id.items()}

# Load token IDs
token_data = torch.load(
    DATA_DIR / "token_ids.pt",
    weights_only=True
)

train_ids = token_data["train"]
val_ids = token_data["validation"]
test_ids = token_data["test"]

VOCAB_SIZE = len(word_to_id)

print("Vocabulary size:", VOCAB_SIZE)
print("Training tokens:", len(train_ids))

Vocabulary size: 10000
Training tokens: 2051910


In [2]:
start_time = time.perf_counter()

unigram_counts = Counter(train_ids)

bigram_counts = Counter(
    zip(train_ids[:-1], train_ids[1:])
)

trigram_counts = Counter(
    zip(train_ids[:-2], train_ids[1:-1], train_ids[2:])
)

counting_time = time.perf_counter() - start_time

print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))

print(f"\nCounting time: {counting_time:.2f} seconds")

Unique unigrams: 9999
Unique bigrams: 433163
Unique trigrams: 1200387

Counting time: 1.28 seconds


In [3]:
def decode_ngram(ngram):
    return " ".join(id_to_word[i] for i in ngram)

print("Top 10 bigrams:\n")

for bigram, count in bigram_counts.most_common(10):
    print(f"{decode_ngram(bigram):25s} {count}")

print("\nTop 10 trigrams:\n")

for trigram, count in trigram_counts.most_common(10):
    print(f"{decode_ngram(trigram):35s} {count}")

Top 10 bigrams:

<unk> ,                   20339
= =                       17816
of the                    17284
<unk> <unk>               16345
the <unk>                 15085
. the                     13156
<unk> .                   12352
in the                    11800
, and                     11675
<unk> and                 8925

Top 10 trigrams:

= = =                               7254
. = =                               4604
<unk> , <unk>                       3113
<unk> and <unk>                     2978
<unk> <unk> ,                       2837
<unk> , and                         2481
<unk> . the                         2244
of the <unk>                        2049
, <unk> ,                           2005
the <unk> of                        1774


In [4]:
# Interpolation weights
LAMBDA_1 = 0.1
LAMBDA_2 = 0.3
LAMBDA_3 = 0.6

TOTAL_TOKENS = sum(unigram_counts.values())

# Small smoothing constant
ALPHA = 0.1

# Padding is not a valid predicted word
PREDICTABLE_VOCAB_SIZE = VOCAB_SIZE - 1

def unigram_prob(w):
    return (
        unigram_counts[w] + ALPHA
    ) / (
        TOTAL_TOKENS + ALPHA * PREDICTABLE_VOCAB_SIZE
    )


def bigram_prob(w_prev, w):
    numerator = bigram_counts[(w_prev, w)]

    denominator = unigram_counts[w_prev]

    if denominator == 0:
        return unigram_prob(w)

    return numerator / denominator


def trigram_prob(w_prev2, w_prev1, w):
    numerator = trigram_counts[(w_prev2, w_prev1, w)]

    denominator = bigram_counts[(w_prev2, w_prev1)]

    if denominator == 0:
        return bigram_prob(w_prev1, w)

    return numerator / denominator


def interpolated_prob(w_prev2, w_prev1, w):

    p1 = unigram_prob(w)
    p2 = bigram_prob(w_prev1, w)
    p3 = trigram_prob(w_prev2, w_prev1, w)

    return (
        LAMBDA_1 * p1
        + LAMBDA_2 * p2
        + LAMBDA_3 * p3
    )

In [5]:
context = ["of", "the"]
target = "game"

w1 = word_to_id[context[0]]
w2 = word_to_id[context[1]]
w3 = word_to_id[target]

print("Context:", context)
print("Target:", target)

print("\nUnigram probability:", unigram_prob(w3))
print("Bigram probability:", bigram_prob(w2, w3))
print("Trigram probability:", trigram_prob(w1, w2, w3))

print("\nInterpolated probability:",
      interpolated_prob(w1, w2, w3))

Context: ['of', 'the']
Target: game

Unigram probability: 0.0010117833227848917
Bigram probability: 0.007264774256698887
Trigram probability: 0.00983568618375376

Interpolated probability: 0.008182022319540412


In [6]:
PAD_ID = word_to_id["<pad>"]

def interpolated_prob(w_prev2, w_prev1, w):
    if w == PAD_ID:
        return 0.0

    p1 = unigram_prob(w)

    # Bigram component
    bigram_context_count = unigram_counts[w_prev1]

    if bigram_context_count > 0:
        p2 = bigram_counts[(w_prev1, w)] / bigram_context_count
    else:
        p2 = 0.0

    # Trigram component
    trigram_context_count = bigram_counts[(w_prev2, w_prev1)]

    if trigram_context_count > 0:
        p3 = trigram_counts[(w_prev2, w_prev1, w)] / trigram_context_count
    else:
        p3 = 0.0

    # Redistribute weights if context is unavailable
    weight1 = LAMBDA_1
    weight2 = LAMBDA_2 if bigram_context_count > 0 else 0.0
    weight3 = LAMBDA_3 if trigram_context_count > 0 else 0.0

    total_weight = weight1 + weight2 + weight3

    return (
        weight1 * p1 +
        weight2 * p2 +
        weight3 * p3
    ) / total_weight

In [7]:
import math
import time

def calculate_perplexity(token_ids):
    total_log_prob = 0.0
    total_predictions = 0

    for i in range(2, len(token_ids)):
        w1 = token_ids[i - 2]
        w2 = token_ids[i - 1]
        target = token_ids[i]

        probability = interpolated_prob(w1, w2, target)

        total_log_prob += math.log(probability)
        total_predictions += 1

    avg_loss = -total_log_prob / total_predictions
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


start_time = time.perf_counter()

val_loss, val_ppl = calculate_perplexity(val_ids)

evaluation_time = time.perf_counter() - start_time

print(f"Validation cross-entropy: {val_loss:.4f}")
print(f"Validation perplexity: {val_ppl:.2f}")
print(f"Evaluation time: {evaluation_time:.2f} seconds")

Validation cross-entropy: 5.2245
Validation perplexity: 185.77
Evaluation time: 0.45 seconds


In [8]:
import pandas as pd

results = []

weight_combinations = [
    (0.1, 0.3, 0.6),
    (0.1, 0.4, 0.5),
    (0.2, 0.3, 0.5),
    (0.2, 0.4, 0.4),
    (0.3, 0.3, 0.4),
    (0.05, 0.25, 0.70),
    (0.05, 0.15, 0.80)
]

for l1, l2, l3 in weight_combinations:

    LAMBDA_1 = l1
    LAMBDA_2 = l2
    LAMBDA_3 = l3

    loss, ppl = calculate_perplexity(val_ids)

    results.append({
        "lambda1": l1,
        "lambda2": l2,
        "lambda3": l3,
        "val_loss": loss,
        "val_ppl": ppl
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "val_ppl"
).reset_index(drop=True)

print(results_df.to_string(index=False))

 lambda1  lambda2  lambda3  val_loss    val_ppl
    0.20     0.40      0.4  5.132142 169.379532
    0.30     0.30      0.4  5.143160 171.256057
    0.20     0.30      0.5  5.157933 173.804868
    0.10     0.40      0.5  5.191707 179.775187
    0.10     0.30      0.6  5.224509 185.769983
    0.05     0.25      0.7  5.339139 208.333338
    0.05     0.15      0.8  5.423585 226.690375


In [9]:
best = results_df.iloc[0]

LAMBDA_1 = float(best["lambda1"])
LAMBDA_2 = float(best["lambda2"])
LAMBDA_3 = float(best["lambda3"])

print("Best interpolation weights:")
print(f"Unigram: {LAMBDA_1}")
print(f"Bigram:  {LAMBDA_2}")
print(f"Trigram: {LAMBDA_3}")

print(f"\nBest validation perplexity: {best['val_ppl']:.2f}")

Best interpolation weights:
Unigram: 0.2
Bigram:  0.4
Trigram: 0.4

Best validation perplexity: 169.38


In [10]:
import numpy as np

def generate_ngram(prompt, max_new_tokens=30,
                   temperature=1.0, top_k=20, seed=42):

    rng = np.random.default_rng(seed)

    tokens = prompt.lower().split()

    if len(tokens) < 2:
        raise ValueError("Prompt must contain at least 2 tokens.")

    generated = tokens.copy()

    for _ in range(max_new_tokens):

        w1 = word_to_id.get(generated[-2], UNK_ID)
        w2 = word_to_id.get(generated[-1], UNK_ID)

        # Calculate probabilities for every vocabulary word
        probs = np.array([
            interpolated_prob(w1, w2, w)
            for w in range(VOCAB_SIZE)
        ])

        # Temperature scaling
        probs = np.power(
            np.maximum(probs, 1e-12),
            1.0 / temperature
        )

        probs[PAD_ID] = 0.0

        # Keep only top-k candidates
        if top_k is not None:
            top_indices = np.argpartition(
                probs, -top_k
            )[-top_k:]

            filtered = np.zeros_like(probs)
            filtered[top_indices] = probs[top_indices]
            probs = filtered

        probs = probs / probs.sum()

        next_id = rng.choice(VOCAB_SIZE, p=probs)
        next_word = id_to_word[int(next_id)]

        generated.append(next_word)

    return " ".join(generated)

In [13]:
PAD_ID = word_to_id["<pad>"]
UNK_ID = word_to_id["<unk>"]

print("PAD_ID:", PAD_ID)
print("UNK_ID:", UNK_ID)

prompts = [
    "the game",
    "in the",
    "the history of"
]

for prompt in prompts:
    print("=" * 60)
    print("PROMPT:", prompt)

    generated_text = generate_ngram(
        prompt=prompt,
        max_new_tokens=30,
        temperature=1.0,
        top_k=20,
        seed=42
    )

    print("GENERATED:")
    print(generated_text)
    print()

PAD_ID: 0
UNK_ID: 1
PROMPT: the game
GENERATED:
the game 's , who had <unk> into a small <unk> . the film was completed in the and <unk> " of a <unk> ( 1 @,@ <unk> of <unk> <unk> in

PROMPT: in the
GENERATED:
in the world in 2015 of <unk> ( a group , and the second world war " , a <unk> @-@ , was , however , it a " <unk> <unk> in

PROMPT: the history of
GENERATED:
the history of . in may 2009 , when he also <unk> . the film was completed in the and <unk> " of a <unk> ( 1 @,@ <unk> of <unk> <unk> in



In [14]:
# Restore the best interpolation weights
LAMBDA_1 = 0.20
LAMBDA_2 = 0.40
LAMBDA_3 = 0.40

CONTEXT_LENGTH = 32

def calculate_aligned_perplexity(token_ids):
    total_log_prob = 0.0
    total_predictions = 0

    num_chunks = (len(token_ids) - 1) // CONTEXT_LENGTH

    for chunk_idx in range(num_chunks):
        start = chunk_idx * CONTEXT_LENGTH

        for offset in range(CONTEXT_LENGTH):
            target_pos = start + offset + 1

            # Use preceding tokens only
            if target_pos < 2:
                continue

            w1 = token_ids[target_pos - 2]
            w2 = token_ids[target_pos - 1]
            target = token_ids[target_pos]

            probability = interpolated_prob(w1, w2, target)

            total_log_prob += math.log(probability)
            total_predictions += 1

    avg_loss = -total_log_prob / total_predictions

    return avg_loss, math.exp(avg_loss), total_predictions


test_loss, test_ppl, num_predictions = calculate_aligned_perplexity(
    test_ids
)

print(f"Test cross-entropy: {test_loss:.4f}")
print(f"Test perplexity: {test_ppl:.2f}")
print(f"Evaluated tokens: {num_predictions}")

Test cross-entropy: 5.0612
Test perplexity: 157.77
Evaluated tokens: 241183


In [16]:
import sys
import time

def get_counter_size(counter):
    size = sys.getsizeof(counter)

    for key, value in counter.items():
        size += sys.getsizeof(key)
        size += sys.getsizeof(value)

        if isinstance(key, tuple):
            for item in key:
                size += sys.getsizeof(item)

    return size


unigram_size = get_counter_size(unigram_counts)
bigram_size = get_counter_size(bigram_counts)
trigram_size = get_counter_size(trigram_counts)

total_size = unigram_size + bigram_size + trigram_size

print("Approximate model memory:")
print(f"Unigrams: {unigram_size / 1024**2:.2f} MB")
print(f"Bigrams:  {bigram_size / 1024**2:.2f} MB")
print(f"Trigrams: {trigram_size / 1024**2:.2f} MB")
print(f"Total:    {total_size / 1024**2:.2f} MB")

# Use a fixed inference workload
NUM_PREDICTIONS = 10000

start_time = time.perf_counter()

for i in range(2, NUM_PREDICTIONS + 2):
    w1 = test_ids[i - 2]
    w2 = test_ids[i - 1]
    target = test_ids[i]

    probability = interpolated_prob(w1, w2, target)

elapsed = time.perf_counter() - start_time

throughput = NUM_PREDICTIONS / elapsed

print(f"Predictions: {NUM_PREDICTIONS}")
print(f"Inference time: {elapsed:.4f} seconds")
print(f"Throughput: {throughput:.2f} tokens/second")

Approximate model memory:
Unigrams: 0.82 MB
Bigrams:  77.83 MB
Trigrams: 241.48 MB
Total:    320.13 MB
Predictions: 10000
Inference time: 0.0220 seconds
Throughput: 453582.62 tokens/second


In [17]:
import json
import pickle
from pathlib import Path

RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# Save model counts and configuration
ngram_model = {
    "unigram_counts": unigram_counts,
    "bigram_counts": bigram_counts,
    "trigram_counts": trigram_counts,
    "lambda1": LAMBDA_1,
    "lambda2": LAMBDA_2,
    "lambda3": LAMBDA_3,
    "alpha": ALPHA,
    "vocab_size": VOCAB_SIZE
}

with open(MODELS_DIR / "ngram_model.pkl", "wb") as f:
    pickle.dump(ngram_model, f)

# Save evaluation results
ngram_results = {
    "model": "Interpolated Trigram",
    "test_loss": 5.0612,
    "test_perplexity": 157.77,
    "model_memory_mb_approx": 320.13,
    "inference_time_seconds": 0.0220,
    "inference_tokens_per_second": 453582.62,
    "inference_tokens": 10000,
    "counting_time_seconds": counting_time,
    "interpolation_weights": [
        LAMBDA_1, LAMBDA_2, LAMBDA_3
    ]
}

with open(RESULTS_DIR / "ngram_results.json", "w") as f:
    json.dump(ngram_results, f, indent=2)

print("N-gram model and results saved successfully!")

N-gram model and results saved successfully!
